<a href="https://colab.research.google.com/github/bhavya0701585/Machine-Learning/blob/main/EXP3_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import math
from collections import Counter

data = {
    "Outlook": ["Sunny", "Sunny", "Overcast", "Rain",
                "Rain", "Rain", "Overcast", "Sunny",
                "Sunny", "Rain", "Sunny", "Overcast",
                "Overcast", "Rain"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool",
                    "Cool", "Cool", "Mild", "Cool", "Mild",
                    "Mild", "Mild", "Hot", "Mild"],
    "Humidity": ["High", "High", "High", "High", "Normal",
                 "Normal", "Normal", "High", "Normal",
                 "Normal", "Normal", "High", "Normal", "High"],
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Weak",
             "Strong", "Strong", "Weak", "Weak", "Weak",
             "Strong", "Strong", "Weak", "Strong"],
    "Play": ["No", "No", "Yes", "Yes", "Yes", "No",
             "Yes", "No", "Yes", "Yes", "Yes", "Yes",
             "Yes", "No"]
}

df = pd.DataFrame(data)

def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    return -sum((n / total) * math.log2(n / total)
                for n in counts.values())

def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])

    return total_entropy - weighted_entropy

def id3(df, attributes, target):
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if not attributes:
        return df[target].mode()[0]

    best = max(attributes,
               key=lambda a: information_gain(df, a, target))

    tree = {best: {}}

    for value in df[best].unique():
        subset = df[df[best] == value]

        if subset.empty:
            tree[best][value] = df[target].mode()[0]
        else:
            remaining = [a for a in attributes if a != best]
            tree[best][value] = id3(subset, remaining, target)

    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)

attributes = ["Outlook", "Temperature", "Humidity", "Wind"]
tree = id3(df, attributes, "Play")

print("Decision Tree:")
print(tree)

sample = {
    "Outlook": "Sunny",
    "Temperature": "Cool",
    "Humidity": "High",
    "Wind": "Strong"
}

print("\nNew Sample:", sample)
print("Predicted Class:", predict(tree, sample))


Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class: No
